# Neural Networks & Deep Learning | Project 1

## Bank Customer Churn Modeling

In [1]:
import tensorflow as tf
tf.set_random_seed(42)
import numpy as np

In [2]:
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import TensorBoard
from keras.callbacks import ModelCheckpoint
from keras.optimizers import SGD

Using TensorFlow backend.


In [3]:
import keras
from keras import backend as K

In [4]:
import numpy as np
import pandas as pd
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# importing ploting libraries
import matplotlib.pyplot as plt   
#importing seaborn for statistical plots
import seaborn as sns
# To enable plotting graphs in Jupyter notebook
%matplotlib inline 
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, f1_score, auc



#### Read data

In [5]:
df = pd.read_csv("bank (1).csv")

In [6]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Drop RowNumber,CustomerId and Surname as these are unique customer attributes and not useful for modeling

In [7]:
df=df.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [8]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [10]:
df['Geography'].value_counts()
df['Exited'].value_counts()
df['Gender'].value_counts()


France     5014
Germany    2509
Spain      2477
Name: Geography, dtype: int64

0    7963
1    2037
Name: Exited, dtype: int64

Male      5457
Female    4543
Name: Gender, dtype: int64

In [11]:
df.isna().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

No Missing values

In [12]:
df['Gender'] = df['Gender'].replace({'Male': 0,'Female': 1})
df['Gender'].value_counts()

0    5457
1    4543
Name: Gender, dtype: int64

In [13]:
df1=df.drop('Geography',axis=1)

In [14]:
df1_get=pd.get_dummies(df['Geography'], drop_first=True)
df1_get.head()

,Germany,Spain
0,0,0
1,0,1
2,0,0
3,0,0
4,0,1


In [15]:
df_new=pd.concat([df1,df1_get],axis=1)
df_new.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Germany,Spain
0,619,1,42,2,0.00,1,1,1,101348.88,1,0,0
1,608,1,41,1,83807.86,1,0,1,112542.58,0,0,1
2,502,1,42,8,159660.80,3,1,0,113931.57,1,0,0
3,699,1,39,1,0.00,2,0,0,93826.63,0,0,0
4,850,1,43,2,125510.82,1,1,1,79084.10,0,0,1


In [16]:
# Splitting data into test and train

X = df_new.drop("Exited", axis=1)
y = df_new["Exited"]

test_size = 0.30 # taking 70:30 training and test set
seed = 7  # Random numbmer seeding for reapeatability of the code
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)


/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/data.py:645: DataConversionWarning: Data with input dtype uint8, int64, float64 were all converted to float64 by StandardScaler.
  return self.partial_fit(X, y)
/anaconda3/lib/python3.7/site-packages/sklearn/base.py:464: DataConversionWarning: Data with input dtype uint8, int64, float64 were all converted to float64 by StandardScaler.
  return self.fit(X, **fit_params).transform(X)
/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/data.py:645: DataConversionWarning: Data with input dtype uint8, int64, float64 were all converted to float64 by StandardScaler.
  return self.partial_fit(X, y)
/anaconda3/lib/python3.7/site-packages/sklearn/base.py:464: DataConversionWarning: Data with input dtype uint8, int64, float64 were all converted to float64 by StandardScaler.
  return self.fit(X, **fit_params).transform(X)


In [18]:
print('Number of examples: ', X_train.shape[0])
print('Number of features for each example: ', X_train.shape[1])

Number of examples:  7000
Number of features for each example:  11


### Initital model

In [19]:
# create model
model = Sequential()
model.add(Dense(11, input_dim=11, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
# Compile model
model.compile(loss='binary_crossentropy', optimizer='sgd', metrics=['accuracy'])

W0813 00:09:33.548259 4364264896 deprecation_wrapper.py:119] From /anaconda3/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0813 00:09:33.551666 4364264896 deprecation_wrapper.py:119] From /anaconda3/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0813 00:09:33.561161 4364264896 deprecation_wrapper.py:119] From /anaconda3/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:4138: The name tf.random_uniform is deprecated. Please use tf.random.uniform instead.

W0813 00:09:33.642868 4364264896 deprecation_wrapper.py:119] From /anaconda3/lib/python3.7/site-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0813 00:09:33.673998 4364264896 deprecation_wrapper.py:119] From /anaconda3

In [20]:
# Fit the model
model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled,y_test), epochs=10, batch_size=70)

W0813 00:09:33.936538 4364264896 deprecation_wrapper.py:119] From /anaconda3/lib/python3.7/site-packages/keras/backend/tensorflow_backend.py:986: The name tf.assign_add is deprecated. Please use tf.compat.v1.assign_add instead.



Train on 7000 samples, validate on 3000 samples
Epoch 1/10
7000/7000 [==============================] - 0s 65us/step - loss: 0.6025 - acc: 0.7594 - val_loss: 0.5590 - val_acc: 0.7983
Epoch 2/10
7000/7000 [==============================] - 0s 27us/step - loss: 0.5460 - acc: 0.7954 - val_loss: 0.5212 - val_acc: 0.7983
Epoch 3/10
7000/7000 [==============================] - 0s 24us/step - loss: 0.5199 - acc: 0.7954 - val_loss: 0.5018 - val_acc: 0.7983
Epoch 4/10
7000/7000 [==============================] - 0s 24us/step - loss: 0.5048 - acc: 0.7954 - val_loss: 0.4890 - val_acc: 0.7983
Epoch 5/10
7000/7000 [==============================] - 0s 24us/step - loss: 0.4939 - acc: 0.7954 - val_loss: 0.4794 - val_acc: 0.7983
Epoch 6/10
7000/7000 [==============================] - 0s 25us/step - loss: 0.4851 - acc: 0.7954 - val_loss: 0.4714 - val_acc: 0.7983
Epoch 7/10
7000/7000 [==============================] - 0s 27us/step - loss: 0.4776 - acc: 0.7954 - val_loss: 0.4647 - val_acc: 0.7983
Epoch 8

In [21]:
y_predict=model.predict_classes(X_test_scaled)

In [22]:
accuracy_score(y_test, y_predict)*100

0.798

### Optimize model

In [34]:
# create model
model = Sequential()
model.add(Dense(11, input_dim=11, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(6, activation='relu'))
model.add(Dense(1, activation='sigmoid'))


In [39]:
opt=SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
#opt=tf.keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False)
opt

In [40]:
X_test_scaled.shape

(3000, 11)

In [41]:
# Compile model
model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])

# Fit the model
model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled,y_test), epochs=10, batch_size=70)

Train on 7000 samples, validate on 3000 samples
Epoch 1/10
7000/7000 [==============================] - 1s 93us/step - loss: 0.4681 - acc: 0.7954 - val_loss: 0.4514 - val_acc: 0.7983
Epoch 2/10
7000/7000 [==============================] - 0s 30us/step - loss: 0.4498 - acc: 0.7954 - val_loss: 0.4374 - val_acc: 0.7983
Epoch 3/10
7000/7000 [==============================] - 0s 33us/step - loss: 0.4374 - acc: 0.7954 - val_loss: 0.4292 - val_acc: 0.7983
Epoch 4/10
7000/7000 [==============================] - 0s 34us/step - loss: 0.4293 - acc: 0.7969 - val_loss: 0.4238 - val_acc: 0.8070
Epoch 5/10
7000/7000 [==============================] - 0s 32us/step - loss: 0.4224 - acc: 0.8099 - val_loss: 0.4181 - val_acc: 0.8213
Epoch 6/10
7000/7000 [==============================] - 0s 60us/step - loss: 0.4170 - acc: 0.8196 - val_loss: 0.4143 - val_acc: 0.8303
Epoch 7/10
7000/7000 [==============================] - 0s 39us/step - loss: 0.4112 - acc: 0.8269 - val_loss: 0.4089 - val_acc: 0.8357
Epoch 8

### Test accuracy with 0.5 probability threshold

In [46]:
y_pred=model.predict_classes(X_test_scaled)
y_pred=y_pred>0.5
y_pred.shape


(3000, 1)

In [50]:
# evaluate the model
scores = model.evaluate(X_test_scaled, y_test)
scores[1]*100
scores

3000/3000 [==============================] - 0s 33us/step


84.63333333333334

[0.3861614541212718, 0.8463333333333334]

In [53]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("Accuracy = ", accuracy_score(y_test, y_pred)*100)

[[2314   81]
 [ 380  225]]
Accuracy =  84.63333333333334


In [43]:
# evaluate the model
scores = model.evaluate(X_test_scaled, y_test)
scores[1]*100
scores



3000/3000 [==============================] - 0s 36us/step


84.63333333333334

[0.3861614541212718, 0.8463333333333334]